# Module 2 — Query your MyGPT library through the API

**Time:** 20 min · **Needs:** module 1 library, `.env` filled in

You will:
1. Connect and list libraries
2. Run the RAG steps one at a time: **retrieve → generate → score**
3. Run a batch of questions and save the results for module 3

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable whether Jupyter was started from the root or this folder
ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from common.mygpt_client import MyGPTClient, outputs_dir, load_fallback_pubmedqa  # noqa: E402
OUT = outputs_dir()

## 1. Connect

In [ ]:
client = MyGPTClient.from_env()       # signs in with MYGPT_USERNAME / MYGPT_PASSWORD
print("Backend:", client.base_url, "| signed in:", bool(client.access_token))
print("Library:", client.dataset, "| LLM:", client.llm, "| embeddings:", client.embedding_model)

import pandas as pd
pd.DataFrame(client.list_datasets())

In [ ]:
docs = client.get_documents()
print(docs.get("dataset_type"), "-", len(docs.get("documents", [])), "documents")
pd.DataFrame(docs.get("documents", [])).head(10)

## 2. RAG step by step

### 2a. Retrieve — `POST /api/get_context/`
Returns the retrieved chunks (`sources`) and the **question relevance score (QRS)** with its parts:
semantic, keyword (BM25) and rerank.

In [ ]:
question = "What is version control and why is it useful for research?"   # ← try your own

ctx = client.get_context(question, max_chunks=5)
print({k: ctx.get(k) for k in ["relevance_score", "semantic_score", "keyword_score", "rerank_score"]})
for s in (ctx.get("sources") or [])[:5]:
    print("-", {k: v for k, v in s.items() if k != "context"})

### 2b. Generate — `POST /api/ollama_generate/`

In [ ]:
from common.mygpt_client import RAG_PROMPT

context_text = ctx.get("context") or "\n\n".join(s.get("context", "") for s in ctx.get("sources", []))
answer = client.generate(RAG_PROMPT.format(context=context_text, question=question))
print(answer)

### 2c. Score — `POST /api/save_answer/`
MyGPT compares the answer with the retrieved chunks, which gives the **answer relevance score (ARS)** and the
**hallucination index (HI)**.

In [ ]:
scores = client.score_answer(question, answer)
{k: scores.get(k) for k in ["relevance_score", "semantic_score", "keyword_score",
                            "hallucination_index_by_equation", "hallucination_index_by_ml"]}

## 3. All in one call: `client.ask()`

In [ ]:
row = client.ask("What is the boiling point of tungsten?")   # off-topic control question
{k: row[k] for k in ["answer", "QRS", "ARS", "HI"]}

**Discuss:** how do QRS and HI for the off-topic question compare with the on-topic one above?

## 4. Batch query and save

In [ ]:
questions = pd.read_csv(
    ROOT
    / "data"
    / "BioASQ"
    / "questions_final.csv"
)["questions"].head(5).tolist()
# Remove .head(5) to query all 163 BioASQ questions.

rows = []
for i, q in enumerate(questions, 1):
    print(f"[{i}/{len(questions)}] {q}")
    try:
        rows.append(client.ask(q))
    except Exception as exc:
        print("   failed:", exc)
        rows.append({"question": q, "error": str(exc)})

results = pd.DataFrame(rows)
results.drop(columns=["context"], errors="ignore").to_csv(OUT / "02_query_results.csv", index=False)
results[["question", "QRS", "ARS", "HI"]]

In [ ]:
import matplotlib.pyplot as plt

plot = results.set_index(results["question"].str.slice(0, 40))[["QRS", "ARS", "HI"]].astype(float)
plot.plot.barh(figsize=(8, 0.5 * len(plot) + 1))
plt.xlabel("score (0-100)")
plt.title("Confidence scores per question")
plt.tight_layout()

## Exercises
1. Rerun the batch with `client.ask(q, max_chunks=10)`. Do ARS and HI change?
2. Set `client.dataset = "workshop-library-b"` (your module 1 comparison library) and compare the two CSVs.
3. Which question has a **high QRS but a poor answer**? Keep it in mind, because module 3 measures this systematically.